**EXA_PY**

In [1]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from exa_py import Exa
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Load environment variables
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
EXA_API_KEY = os.getenv("EXA_API_KEY")

# Initialize Exa
exa = Exa(api_key=os.environ["EXA_API_KEY"])

# Define Exa tools
@tool
def search_and_contents(query: str):
    """Search for webpages based on the query and retrieve their contents."""
    response = exa.search_and_contents(
        query, use_autoprompt=True, num_results=5, text=True, highlights=True
    )
    # Format response as a string
    formatted_results = "\n".join([
        f"Title: {result.title}\nURL: {result.url}\nText: {result.text}\nHighlights: {result.highlights}"
        for result in response.results
    ])
    return formatted_results or "No results found."

@tool
def find_similar_and_contents(url: str):
    """Search for webpages similar to a given URL and retrieve their contents.
    The url passed in should be a URL returned from `search_and_contents`.
    """
    try:
        response = exa.find_similar_and_contents(url, num_results=5, text=True, highlights=True)
        # Format response as a string
        formatted_results = "\n".join([
            f"Title: {result.title}\nURL: {result.url}\nText: {result.text}\nHighlights: {result.highlights}"
            for result in response.results
        ])
        return formatted_results or "No similar results found."
    except Exception as e:
        return f"Error finding similar content: {str(e)}"

tools = [search_and_contents, find_similar_and_contents]



d:\Ashok\AI\2082_projects\PRACTISES\Travel_GURU\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**RAG_RESPONSE**

In [3]:
import sys
from pathlib import Path
#get current working directoy
parent_dir = sys.path.append(str(Path().cwd().parent))

from langchain_pinecone import PineconeVectorStore
from src.helpers import load_hugging_face_embeddings
from pinecone import Pinecone

from langchain.tools import tool

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY)
print(PINECONE_API_KEY)

index_name = "travel-guru"

@tool
def search_pdf_docs(query:str) -> str:
    """Query pdf documents indexed in Pinecone"""
    embeddings = load_hugging_face_embeddings()
    vectorstore = PineconeVectorStore(
        index_name=index_name,
        embedding=embeddings
    )
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 5}
    )
    docs = retriever.get_relevant_documents(query)
    return "\n\n".join([d.page_content for d in docs]) or "No results found"


query = "What are the routes to the Everest Base Camp?"
raw_rag_response = search_pdf_docs.invoke({"query": query})
print(raw_rag_response)

pcsk_52fsWW_PyGokwzc1GSCGMNuJ5rEueegbVL8sXShu6MSvcivuUM17LvrtJwLS3Kn2ffUntz


d:\Ashok\AI\2082_projects\PRACTISES\Travel_GURU\src\helpers.py:26: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
C:\Users\ashok\AppData\Local\Temp\ipykernel_20552\3075682805.py:30: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)


e x p e r i e n c e . 
E v e r e s t B a s e C a m p T r e k 
Everest Base Camp T rek Itinerary 
D a y 1 : F l i g h t f r o m K a t h m a n d u t o L u k l a a n d T r e k t o P h a k d i n g 
2 4

TRIP FACTSEVEREST BASE CAMP TREK
Everest Base Camp Trek is the busiest trail in 
Khumbu and, after a short ﬂight from 
Kathmandu, you can start your trek. A ﬁrst 
acclimatization day is normally set at Namche, 
during which you can explore the Sherpa 
Museum, Syangboche Airport, Everest View 
Hotel and Edmund Hillary Memorial, as well as 
the historic twin villages of Khumjung and 
Khunde. There is a dramatic rise in altitude, so it 
is recommended to have at least two

Thamel, Kathmandu,
Nepal P .O.Box: 4003
+977-9841773981
+977-9851139218
info@magicalnepal.com
www.magicalnepal.com
1 WEEK TREK
› Everest Panorama
› Trek
 Helambu Trek
› Ghorepani Poonhill Trek
2 WEEK TREK
› Gokyo Lake Trek
› Everest Base Camp Trek
› Langtang Trek
› Gosaikunda 
Lake Trek
› Tamang Heritage Trek
› Lower Manaslu

In [6]:


# Initialize Gemini 2.0 Flash
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0.3,
    max_output_tokens=16000
)
# Define prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that uses Exa to search the web and find similar content. 
    - Use `search_and_contents` for general queries to find webpages and their contents.
    - Use `find_similar_and_contents` only when asked to find content similar to a specific URL.
    - Summarize results concisely in a ranked list if applicable, or provide a clear answer based on the retrieved information.
    - If no relevant results are found, state so clearly."""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# Bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

# Create agent
agent = create_tool_calling_agent(llm=llm_with_tools, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

query = "What are the routes to the Everest Base Camp?"

try:
    response = agent_executor.invoke({"input": query})
    print("Agent output:\n", response["output"])
except Exception as e:
    print("Agent execution error:", str(e))




> Entering new AgentExecutor chain...

Invoking: `search_and_contents` with `{'query': 'Everest Base Camp routes'}`


Title: What are the different Everest Base Camp trek routes?
URL: https://followalice.com/knowledge/everest-base-camp-route
Text: The Everest Base Camp trek is a truly exhilarating and once-in-a-lifetime experience. There are various routes you can trek there and back, and you can even opt to fly back after reaching base camp.

Here's an outline of each of the four different Everest Base Camp (EBC) trek routes offered by us at [Follow Alice](https://followalice.com/). We discuss the pros of each to help you decide which suits you best in terms of duration, difficulty, cost, variety of scenery and more!

## 1\. Classic EBC trek

![Follow Alice classic EBC trek map](data:image/svg+xml,%3Csvg%20xmlns='http://www.w3.org/2000/svg'%20viewBox='0%200%20800%20493'%3E%3Crect%20height='100%25'%20style='fill:%23e9e9e9'%20width='100%25'%20x='0'%20y='0'%20/%3E%3C/svg%3E)

The tradi

In [5]:
from langchain_core.runnables import RunnablePassthrough
def run_combined_search(query:str)-> str:
    """Run combined search using Exa and Gemini"""
    raw_rag_response = search_pdf_docs(query)
    exa_web_search = search_and_contents(query)
    #extract urls from the web results to find similar content
    urls = [line.split("URL:")[1] for line in exa_web_search.split("\n") if "URL:" in line]
    similar_content = ""
    for url in urls:
        try:
            similar_content += find_similar_and_contents(url)
        except Exception as e:
            print(f"Error finding similar content for {url}: {str(e)}")
    
    return f"""WEB RESULTS:{exa_web_search}, SIMILAR CONTENT:{similar_content}, PDF RESULTS:{raw_rag_response}"""

# 2. Create the summarization prompt (unchanged)
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a synthesis assistant. Combine these information sources into:
    - A concise 3-5 paragraph summary (under 1000 tokens)
    - Key points as bullet points
    - Cite sources with [WEB], [SIMILAR], or [DOC] prefixes
    
    Structure:
    1. Overview
    2. Key Findings
    3. Recommendations"""),
    ("human", "QUERY: {query}\n\nRESULTS: {results}")
])

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0.3,
    max_output_tokens=1000
)

processing_chain = {
    "query": RunnablePassthrough(),
    "results": run_combined_search
} | summarize_prompt | llm

# 4. Example usage
query = "What are the routes to the Everest Base Camp?"
final_output = processing_chain.invoke(query)
print(final_output.content)



C:\Users\ashok\AppData\Local\Temp\ipykernel_20552\1405586520.py:4: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  raw_rag_response = search_pdf_docs(query)


The Everest Base Camp (EBC) trek is a popular and challenging adventure, drawing trekkers from around the globe to the foot of the world's highest mountain [WEB]. The trek to EBC in Nepal offers a mix of cultural experiences, stunning mountain scenery, and a physical test for those who undertake it [WEB]. The classic route begins with a flight from Kathmandu to Lukla, a mountain village with a small airstrip [WEB, PDF]. From there, the trek winds through Sagarmatha National Park, passing Sherpa villages, monasteries, and iconic glaciers, before reaching Everest Base Camp at an altitude of 5,364 meters (17,598 feet) [WEB].

There are several variations to the EBC trek, catering to different preferences and fitness levels [WEB]. The classic EBC trek involves hiking from Lukla to Everest Base Camp and back, covering a total distance of 130 km (81 miles) and taking approximately 12-14 days [WEB, DOC]. Other options include the EBC trek with a helicopter return, which shortens the return jo